# ST446 Project Data Cleaning and Feature Engineering

## Imports and Data Loading

In [1]:
# Package imports for data processing and cleaning
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml import Pipeline
import os

In [2]:
import warnings, logging
warnings.filterwarnings('ignore')

# Suppress FileStreamSink warning specifically
logging.getLogger("org.apache.spark.sql.execution.streaming.FileStreamSink").setLevel(logging.ERROR)
logging.getLogger("py4j").setLevel(logging.ERROR)

spark = SparkSession.builder.appName("Flights_2018_2022").getOrCreate()

# flights_partitioned was loaded in preprocessed.py, so we can skip the raw data loading from zip file
df_raw = spark.read.parquet("data/flights_partitioned/*.parquet")

print("Rows:", df_raw.count(), "Cols:", len(df_raw.columns))
df_raw.printSchema()
display(df_raw.head(5))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/30 18:34:35 WARN Utils: Your hostname, Norahs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 100.65.245.116 instead (on interface en0)
26/03/30 18:34:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 18:34:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 18:34:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/30 18:34:40 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/flights_partitioned/*.parquet.
java.io.FileNotFoundException: File data/flights_partit

Rows: 32062878 Cols: 20
root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER_AIRLINE_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DELAY_NEW: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double (nullable = true)
 |-- NAS_DELAY: double (nullable = true)
 |-- SECURITY_DELAY: double (nullable = true)
 |-- LATE_AIRCRAFT_DELAY: double (nullable = true)



[Row(FL_DATE='12/1/2019 12:00:00 AM', OP_CARRIER_AIRLINE_ID=19393, ORIGIN='ABQ', DEST='AUS', DEP_TIME=1739, DEP_DELAY=24.0, DEP_DELAY_NEW=24.0, ARR_TIME=2012, ARR_DELAY=17.0, ARR_DELAY_NEW=17.0, CANCELLED=0.0, CANCELLATION_CODE=None, DIVERTED=0.0, AIR_TIME=76.0, DISTANCE=619.0, CARRIER_DELAY=4.0, WEATHER_DELAY=0.0, NAS_DELAY=0.0, SECURITY_DELAY=0.0, LATE_AIRCRAFT_DELAY=13.0),
 Row(FL_DATE='12/1/2019 12:00:00 AM', OP_CARRIER_AIRLINE_ID=19393, ORIGIN='ABQ', DEST='BWI', DEP_TIME=1223, DEP_DELAY=23.0, DEP_DELAY_NEW=23.0, ARR_TIME=1736, ARR_DELAY=1.0, ARR_DELAY_NEW=1.0, CANCELLED=0.0, CANCELLATION_CODE=None, DIVERTED=0.0, AIR_TIME=177.0, DISTANCE=1670.0, CARRIER_DELAY=None, WEATHER_DELAY=None, NAS_DELAY=None, SECURITY_DELAY=None, LATE_AIRCRAFT_DELAY=None),
 Row(FL_DATE='12/1/2019 12:00:00 AM', OP_CARRIER_AIRLINE_ID=19393, ORIGIN='ABQ', DEST='DAL', DEP_TIME=601, DEP_DELAY=-4.0, DEP_DELAY_NEW=0.0, ARR_TIME=831, ARR_DELAY=-14.0, ARR_DELAY_NEW=0.0, CANCELLED=0.0, CANCELLATION_CODE=None, DIVERTE

## Data Cleaning 

In [3]:
# 1. Filter scope and drop key nulls
df = (
    df_raw.filter(col("DEST").isin("ORD", "JFK", "ATL"))
      .filter((col("CANCELLED") == 0) & (col("DIVERTED") == 0))
      .dropna(subset=["FL_DATE", "DEP_TIME", "ARR_DELAY", "DISTANCE"])
)

# 2. Standardize column names and types
df = df.withColumn(
    "FL_TS",
    expr("try_to_timestamp(FL_DATE, 'M/d/yyyy h:mm:ss a')")
)
# 2. Feature engineering
df = df.withColumn("FL_DATE", to_date(col("FL_TS"), "M/d/yyyy"))
df = df.withColumn("day_of_week", dayofweek(col("FL_DATE"))) \
       .withColumn("year", year(col("FL_DATE"))) \
       .withColumn("month", month(col("FL_DATE"))) \
       .withColumn("dep_hour", floor(col("DEP_TIME") / 100).cast("int")) 

# 3. Create distance bucket
df = df.withColumn(
    "distance_group",
    when(col("DISTANCE") <= 500, 0)
    .when(col("DISTANCE") <= 1000, 1)
    .when(col("DISTANCE") <= 2000, 2)
    .otherwise(3).cast(IntegerType())
)

# 4. Create indicators for delay causes (treat nulls as 0)
delay_cols = ["CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
              "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]
for c in delay_cols:
    df = df.withColumn(c, coalesce(col(c), lit(0.0)))

# for each delay type, create a binary indicator if that delay was > 0
# if multiple delay types > 0, they will all be marked as 1
df = (
    df.withColumn("delay_carrier", (col("CARRIER_DELAY") > 0).cast("int"))
      .withColumn("delay_weather", (col("WEATHER_DELAY") > 0).cast("int"))
      .withColumn("delay_nas", (col("NAS_DELAY") > 0).cast("int"))
      .withColumn("delay_security", (col("SECURITY_DELAY") > 0).cast("int"))
      .withColumn("delay_late_aircraft", (col("LATE_AIRCRAFT_DELAY") > 0).cast("int"))
      .withColumn("delay_none", (col("ARR_DELAY") <= 0).cast("int"))
)

# 5. Create a multiclass target variable based on ARR_DELAY
df = df.withColumn(
    "delay_class",
    when(col("ARR_DELAY") <= 0, 0)      # no delay / early
    .when(col("ARR_DELAY") <= 60, 1)    # minor
    .when(col("ARR_DELAY") <= 120, 2)   # major
    .otherwise(3).cast("int")           # severe
)

# 6. Rename columns to lowercase
df = df.select([
    col(c).alias(c.lower()) for c in df.columns
])
df = df.withColumnRenamed("op_carrier_airline_id", "carrier_id")

## Final Dataset Creation

In [ ]:
# Final column selection for modeling
final_cols = [
    'carrier_id', 'origin', 'dest', 'dep_hour', 'year', 'day_of_week', 
    'month', 'distance', 'distance_group', 'arr_delay',
    'delay_carrier', 'delay_weather', 'delay_nas', 
    'delay_security', 'delay_late_aircraft', 'delay_none',
    'delay_class'  # target
]


df_final = df.select(final_cols)
print("Selected columns:", df_final.columns)
print("Final dataset rows:", df_final.count())
df_final.show(5)

Selected columns: ['carrier_id', 'origin', 'dest', 'dep_hour', 'year', 'day_of_week', 'month', 'distance', 'distance_group', 'arr_delay', 'delay_carrier', 'delay_weather', 'delay_nas', 'delay_security', 'delay_late_aircraft', 'delay_none', 'delay_class']


Final dataset rows: 3480548
+----------+------+----+--------+----+-----------+-----+--------+--------------+---------+-------------+-------------+---------+--------------+-------------------+----------+-----------+
|carrier_id|origin|dest|dep_hour|year|day_of_week|month|distance|distance_group|arr_delay|delay_carrier|delay_weather|delay_nas|delay_security|delay_late_aircraft|delay_none|delay_class|
+----------+------+----+--------+----+-----------+-----+--------+--------------+---------+-------------+-------------+---------+--------------+-------------------+----------+-----------+
|     19393|   AUS| ATL|       5|2019|          1|   12|   813.0|             1|    -13.0|            0|            0|        0|             0|                  0|         1|          0|
|     19393|   AUS| ATL|      11|2019|          1|   12|   813.0|             1|     -7.0|            0|            0|        0|             0|                  0|         1|          0|
|     19393|   AUS| ATL|      18|2019

In [5]:
# make a histogram of the target variable
df_final.groupBy("delay_class").count().orderBy("delay_class").show()

+-----------+-------+
|delay_class|  count|
+-----------+-------+
|          0|2444610|
|          1| 829685|
|          2| 114931|
|          3|  91322|
+-----------+-------+



In [ ]:
# Save the cleaned dataset for modeling
df_final.coalesce(4).write.mode("overwrite") \
        .option("compression", "snappy") \
        .parquet("data/df_final_clean.parquet/")



In [1]:
spark.stop()

NameError: name 'spark' is not defined